In [28]:
import boto3
import json
import time
import uuid
import zipfile
from io import BytesIO
import pprint
import logging
print(boto3.__version__)


1.42.44


In [29]:
logging.basicConfig(format='[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s', level=logging.INFO)
logger = logging.getLogger(__name__)

In [78]:
sts_client = boto3.client(service_name = "sts" , region_name = "us-east-1")
iam_client = boto3.client(service_name = "iam", region_name = "us-east-1")
lambda_client = boto3.client(service_name = "lambda" , region_name = "us-east-1")
bedrock_agent_client = boto3.client(service_name= "bedrock-agent",region_name = "us-east-1")
bedrock_agent_runtime_client = boto3.client(service_name = "bedrock-agent-runtime" , region_name= "us-east-1")

In [31]:
session = boto3.Session()
print(session)
region = session.region_name
account_id = sts_client.get_caller_identity()["Account"]
region, account_id

Session(region_name='us-east-1')


('us-east-1', '546203070602')

In [32]:
inference_profile = "us.amazon.nova-micro-v1:0"
foundation_model = inference_profile[3:]
foundation_model

'amazon.nova-micro-v1:0'

In [33]:
suffix = f"{region}-{account_id}"
agent_name = "hr-assistant-function-def"
agent_bedrock_allow_policy_name = f"{agent_name}-ba-{suffix}"
agent_role_name = f'AmazonBedrockExecutionRoleForAgents_{agent_name}'
agent_description = "Agent for providing HR assistance to manage vacation time"
agent_instruction = "You are an HR agent, helping employees understand HR policies and manage vacation time"
agent_action_group_name = "VacationsActionGroup"
agent_action_group_description = "Actions for getting the number of available vacations days for an employee and confirm new time off"
agent_alias_name = f"{agent_name}-alias"
lambda_function_role = f'{agent_name}-lambda-role-{suffix}'
lambda_function_name = f'{agent_name}-{suffix}'

In [34]:
# creating employee database to be used by lambda function
import sqlite3
import random
from datetime import date, timedelta

# Connect to the SQLite database (creates a new one if it doesn't exist)
conn = sqlite3.connect('employee_database.db')
c = conn.cursor()

# Create the employees table
c.execute('''CREATE TABLE IF NOT EXISTS employees
                (employee_id INTEGER PRIMARY KEY AUTOINCREMENT, employee_name TEXT, employee_job_title TEXT, employee_start_date TEXT, employee_employment_status TEXT)''')

# Create the vacations table
c.execute('''CREATE TABLE IF NOT EXISTS vacations
                (employee_id INTEGER, year INTEGER, employee_total_vacation_days INTEGER, employee_vacation_days_taken INTEGER, employee_vacation_days_available INTEGER, FOREIGN KEY(employee_id) REFERENCES employees(employee_id))''')

# Create the planned_vacations table
c.execute('''CREATE TABLE IF NOT EXISTS planned_vacations
                (employee_id INTEGER, vacation_start_date TEXT, vacation_end_date TEXT, vacation_days_taken INTEGER, FOREIGN KEY(employee_id) REFERENCES employees(employee_id))''')

# Generate some random data for 10 employees
employee_names = ['John Doe', 'Jane Smith', 'Bob Johnson', 'Alice Williams', 'Tom Brown', 'Emily Davis', 'Michael Wilson', 'Sarah Taylor', 'David Anderson', 'Jessica Thompson']
job_titles = ['Manager', 'Developer', 'Designer', 'Analyst', 'Accountant', 'Sales Representative']
employment_statuses = ['Active', 'Inactive']

for i in range(10):
    name = employee_names[i]
    job_title = random.choice(job_titles)
    start_date = date(2015 + random.randint(0, 7), random.randint(1, 12), random.randint(1, 28)).strftime('%Y−%m−%d')
    employment_status = random.choice(employment_statuses)
    c.execute("INSERT INTO employees (employee_name, employee_job_title, employee_start_date, employee_employment_status) VALUES (?, ?, ?, ?)", (name, job_title, start_date, employment_status))
    employee_id = c.lastrowid

    # Generate vacation data for the current employee
    for year in range(date.today().year, date.today().year - 3, -1):
        total_vacation_days = random.randint(10, 30)
        days_taken = random.randint(0, total_vacation_days)
        days_available = total_vacation_days - days_taken
        c.execute("INSERT INTO vacations (employee_id, year, employee_total_vacation_days, employee_vacation_days_taken, employee_vacation_days_available) VALUES (?, ?, ?, ?, ?)", (employee_id, year, total_vacation_days, days_taken, days_available))

        # Generate some planned vacations for the current employee and year
        num_planned_vacations = random.randint(0, 3)
        for _ in range(num_planned_vacations):
            start_date = date(year, random.randint(1, 12), random.randint(1, 28)).strftime('%Y−%m−%d')
            end_date = (date(int(start_date[:4]), int(start_date[5:7]), int(start_date[8:])) + timedelta(days=random.randint(1, 14))).strftime('%Y−%m−%d')
            days_taken = (date(int(end_date[:4]), int(end_date[5:7]), int(end_date[8:])) - date(int(start_date[:4]), int(start_date[5:7]), int(start_date[8:])))
            c.execute("INSERT INTO planned_vacations (employee_id, vacation_start_date, vacation_end_date, vacation_days_taken) VALUES (?, ?, ?, ?)", (employee_id, start_date, end_date, days_taken.days))

# Commit the changes and close the connection
conn.commit()
conn.close()

In [35]:
%%writefile lambda_function.py
import os
import json
import shutil
import sqlite3
from datetime import datetime


def get_available_vacations_days(employee_id):
    conn = sqlite3.connect('/tmp/employee_database.db')
    c = conn.cursor()

    if employee_id:
        c.execute("""
            SELECT employee_vacation_days_available
            FROM vacations
            WHERE employee_id = ?
            ORDER BY year DESC
            LIMIT 1
        """, (employee_id,))

        available_vacation_days = c.fetchone()

        if available_vacation_days:
            available_vacation_days = available_vacation_days[0]
            conn.close()
            return available_vacation_days
        else:
            conn.close()
            return f"No vacation data found for employed_id {employee_id}"
    else:
        conn.close()
        raise Exception("No employee id provided")

Overwriting lambda_function.py


In [37]:
try:
    assume_role_policy_document = {
        "Version":"2012-10-17",
        "Statement":[
            {
                "Effect": "Allow",
                "Principal":{
                    "Service": "lambda.amazonaws.com"
                },
                "Action":"sts:AssumeRole"
            }
        ]
    }
    assume_role_policy_document_json = json.dumps(assume_role_policy_document)
    lambda_iam_role = iam_client.create_role(
        RoleName = lambda_function_role,
        AssumeRolePolicyDocument = assume_role_policy_document_json
    )
    time.sleep(10)
except:
    lambda_iam_role = iam_client.get_role(RoleName=lambda_function_role)

iam_client.attach_role_policy(
    RoleName=lambda_function_role,
    PolicyArn='arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole'
) 

{'ResponseMetadata': {'RequestId': 'b59901cb-a56a-46f2-9fd5-b7d48e75a80a',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Sun, 08 Feb 2026 02:07:13 GMT',
   'x-amzn-requestid': 'b59901cb-a56a-46f2-9fd5-b7d48e75a80a',
   'content-type': 'text/xml',
   'content-length': '212'},
  'RetryAttempts': 0}}

In [71]:
# Package up the code again (ensure you saved lambda_function.py first!)
s = BytesIO()
with zipfile.ZipFile(s, 'w') as z:
    z.write("lambda_function.py")
    z.write("employee_database.db")
zip_content = s.getvalue()

# UPDATE the existing function instead of creating a new one
try:
    response = lambda_client.update_function_code(
        FunctionName=lambda_function_name,
        ZipFile=zip_content
    )
    print("✅ Lambda code updated successfully!")
except lambda_client.exceptions.ResourceNotFoundException:
    print("❌ Function not found. Check your lambda_function_name variable.")




✅ Lambda code updated successfully!


In [72]:
bedrock_agent_bedrock_allow_policy_statement = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "AmazonBedrockAgentBedrockFoundationModelPolicy",
            "Effect": "Allow",
            "Action": "bedrock:InvokeModel",
            "Resource": [
                f"arn:aws:bedrock:*::foundation-model/{foundation_model}",
                f"arn:aws:bedrock:*:*:inference-profile/{inference_profile}"
            ]
        },
        {
            "Sid": "AmazonBedrockAgentBedrockGetInferenceProfile",
            "Effect": "Allow",
            "Action":  [
                "bedrock:GetInferenceProfile",
                "bedrock:ListInferenceProfiles",
                "bedrock:UseInferenceProfile"
            ],
            "Resource": [
                f"arn:aws:bedrock:*:*:inference-profile/{inference_profile}"
            ]
        }
    ]
}

bedrock_policy_json = json.dumps(bedrock_agent_bedrock_allow_policy_statement)

agent_bedrock_policy = iam_client.create_policy(
    PolicyName=agent_bedrock_allow_policy_name,
    PolicyDocument=bedrock_policy_json
)

EntityAlreadyExistsException: An error occurred (EntityAlreadyExists) when calling the CreatePolicy operation: A policy called hr-assistant-function-def-ba-us-east-1-546203070602 already exists. Duplicate names are not allowed.

In [73]:
# Create IAM Role for the agent and attach IAM policies
assume_role_policy_document = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {
            "Service": "bedrock.amazonaws.com"
        },
        "Action": "sts:AssumeRole"
    }]
}

assume_role_policy_document_json = json.dumps(assume_role_policy_document)
agent_role = iam_client.create_role(
    RoleName=agent_role_name,
    AssumeRolePolicyDocument=assume_role_policy_document_json
)

# Pause to make sure role is created
time.sleep(10)

iam_client.attach_role_policy(
    RoleName=agent_role_name,
    PolicyArn=agent_bedrock_policy['Policy']['Arn']
)

EntityAlreadyExistsException: An error occurred (EntityAlreadyExists) when calling the CreateRole operation: Role with name AmazonBedrockExecutionRoleForAgents_hr-assistant-function-def already exists.

In [74]:
response = bedrock_agent.create_agent(
    agentName = agent_name,
    agentResourceRoleArn=agent_role['Role']['Arn'],
    description = agent_description,
    idleSessionTTLInSeconds = 1800,
    foundationModel = inference_profile,
    instruction = agent_instruction
)
agent_id = response['agent']['agentId']
agent_id , response

ConflictException: An error occurred (ConflictException) when calling the CreateAgent operation: Could not perform Create operation, since the hr-assistant-function-def (id: P58WRUHKR4) with the same name hr-assistant-function-def already exists

In [48]:
agent_functions = [
    {
        'name': 'get_available_vacations_days',
        'description': 'get the number of vacations available for a certain employee',
        'parameters': {
            "employee_id": {
                "description": "the id of the employee to get the available vacations",
                "required": True,
                "type": "integer"
            }
        }
    },
    {
        'name': 'reserve_vacation_time',
        'description': 'reserve vacation time for a specific employee - you need all parameters to reserve vacation time',
        'parameters': {
            "employee_id": {
                "description": "the id of the employee for which time off will be reserved",
                "required": True,
                "type": "integer"
            },
            "start_date": {
                "description": "the start date for the vacation time",
                "required": True,
                "type": "string"
            },
            "end_date": {
                "description": "the end date for the vacation time",
                "required": True,
                "type": "string"
            }
        }
    },
]

In [49]:
agent_action_group_response = bedrock_agent_client.create_agent_action_group(
    agentId = agent_id,
    agentVersion = "DRAFT",
    actionGroupExecutor={
        'lambda': lambda_function['FunctionArn']
    },
    actionGroupName=agent_action_group_name,
    functionSchema={
        'functions': agent_functions
    },
    description=agent_action_group_description
)
agent_action_group_response

{'ResponseMetadata': {'RequestId': '705dd9c9-1edf-4ae9-a184-d904ca8df33d',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Sun, 08 Feb 2026 02:26:53 GMT',
   'content-type': 'application/json',
   'content-length': '1335',
   'connection': 'keep-alive',
   'x-amzn-requestid': '705dd9c9-1edf-4ae9-a184-d904ca8df33d',
   'x-amz-apigw-id': 'YcMhGG6mIAMEU_w=',
   'x-amzn-trace-id': 'Root=1-6987f46d-234086fe39570c2150f886bb'},
  'RetryAttempts': 0},
 'agentActionGroup': {'agentId': 'P58WRUHKR4',
  'agentVersion': 'DRAFT',
  'actionGroupId': 'FBWXSPMPLZ',
  'actionGroupName': 'VacationsActionGroup',
  'description': 'Actions for getting the number of available vacations days for an employee and confirm new time off',
  'createdAt': datetime.datetime(2026, 2, 8, 2, 26, 53, 162351, tzinfo=tzutc()),
  'updatedAt': datetime.datetime(2026, 2, 8, 2, 26, 53, 162351, tzinfo=tzutc()),
  'actionGroupExecutor': {'lambda': 'arn:aws:lambda:us-east-1:546203070602:function:hr-assistant-function-def-us-e

In [75]:
response = lambda_client.add_permission(
    FunctionName = lambda_function_name,
    StatementId = "allow_bedrock",
    Action = "lambda:InvokeFunction",
    Principal = "bedrock.amazonaws.com",
    SourceArn = f"arn:aws:bedrock:{region}:{account_id}:agent/{agent_id}",
)

ResourceConflictException: An error occurred (ResourceConflictException) when calling the AddPermission operation: The statement id (allow_bedrock) provided already exists. Please provide a new statement id, or remove the existing statement.

In [51]:
response = bedrock_agent_client.prepare_agent(
    agentId = agent_id
)
response

{'ResponseMetadata': {'RequestId': '8b2c39a1-c2ad-4397-ad58-e92de56139e4',
  'HTTPStatusCode': 202,
  'HTTPHeaders': {'date': 'Sun, 08 Feb 2026 02:34:28 GMT',
   'content-type': 'application/json',
   'content-length': '119',
   'connection': 'keep-alive',
   'x-amzn-requestid': '8b2c39a1-c2ad-4397-ad58-e92de56139e4',
   'x-amz-apigw-id': 'YcNoQHq5IAMEYmQ=',
   'x-amzn-trace-id': 'Root=1-6987f634-32107ddf4e8ee3c60d233710'},
  'RetryAttempts': 0},
 'agentId': 'P58WRUHKR4',
 'agentStatus': 'PREPARING',
 'agentVersion': 'DRAFT',
 'preparedAt': datetime.datetime(2026, 2, 8, 2, 34, 28, 759531, tzinfo=tzutc())}

In [67]:
response = bedrock_agent_client.create_agent_alias(
    agentId = 'P58WRUHKR4',
    agentAliasName = "test-alias-3"
)
response


{'ResponseMetadata': {'RequestId': 'a3546eee-c833-4a53-b7f7-4bb29312a68e',
  'HTTPStatusCode': 202,
  'HTTPHeaders': {'date': 'Sun, 08 Feb 2026 02:57:12 GMT',
   'content-type': 'application/json',
   'content-length': '382',
   'connection': 'keep-alive',
   'x-amzn-requestid': 'a3546eee-c833-4a53-b7f7-4bb29312a68e',
   'x-amz-apigw-id': 'YcQ9YEKwIAMEN0Q=',
   'x-amzn-trace-id': 'Root=1-6987fb88-05e81bee471750f002db6954'},
  'RetryAttempts': 0},
 'agentAlias': {'agentId': 'P58WRUHKR4',
  'agentAliasId': 'UUXB8ZRCNL',
  'agentAliasName': 'test-alias-3',
  'agentAliasArn': 'arn:aws:bedrock:us-east-1:546203070602:agent-alias/P58WRUHKR4/UUXB8ZRCNL',
  'routingConfiguration': [{}],
  'createdAt': datetime.datetime(2026, 2, 8, 2, 57, 12, 665014, tzinfo=tzutc()),
  'updatedAt': datetime.datetime(2026, 2, 8, 2, 57, 12, 665014, tzinfo=tzutc()),
  'agentAliasStatus': 'CREATING',
  'aliasInvocationState': 'ACCEPT_INVOCATIONS'}}

In [80]:
agent_alias_id = 'UUXB8ZRCNL'
## create a random id for session initiator id
session_id:str = str(uuid.uuid1())
enable_trace:bool = False
end_session:bool = False

# invoke the agent API
agentResponse = bedrock_agent_runtime_client.invoke_agent(
    inputText="How much vacation does the employee with employee_id set to 1 have available?",
    agentId=agent_id,
    agentAliasId=agent_alias_id, 
    sessionId=session_id,
    enableTrace=enable_trace, 
    endSession= end_session
)

logger.info(pprint.pprint(agentResponse))

event_stream = agentResponse['completion']
try:
    for event in event_stream:        
        if 'chunk' in event:
            data = event['chunk']['bytes']
            logger.info(f"Final answer ->\n{data.decode('utf8')}")
            agent_answer = data.decode('utf8')
            end_event_received = True
        elif 'trace' in event:
            logger.info(json.dumps(event['trace'], indent=2))
        else:
            raise Exception("unexpected event.", event)
except Exception as e:
    raise Exception("unexpected event.", e)

[2026-02-07 19:05:13,042] p74989 {378966670.py:17} INFO - None


{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-type': 'application/vnd.amazon.eventstream',
                                      'date': 'Sun, 08 Feb 2026 03:05:13 GMT',
                                      'transfer-encoding': 'chunked',
                                      'x-amz-bedrock-agent-session-id': 'fc486b0e-049a-11f1-aa37-02211ae2cbcb',
                                      'x-amzn-bedrock-agent-content-type': 'application/json',
                                      'x-amzn-requestid': '23e0811f-be2c-4185-8ee8-23dfa74d7054'},
                      'HTTPStatusCode': 200,
                      'RequestId': '23e0811f-be2c-4185-8ee8-23dfa74d7054',
                      'RetryAttempts': 0},
 'completion': <botocore.eventstream.EventStream object at 0x11849b290>,
 'contentType': 'application/json',
 'sessionId': 'fc486b0e-049a-11f1-aa37-02211ae2cbcb'}


Exception: ('unexpected event.', EventStreamError("An error occurred (dependencyFailedException) when calling the InvokeAgent operation: Your request couldn't be completed. Lambda function arn:aws:lambda:us-east-1:546203070602:function:hr-assistant-function-def-us-east-1-546203070602 encountered a problem while processing request.The error message from the Lambda function is Unhandled. Check the Lambda function log for error details, then try your request again after fixing the error."))

In [61]:
agents = bedrock_agent_client.list_agents()
for agent in agents.get('agentSummaries', []):
    print(f"Agent Name: {agent['agentName']} | Agent ID: {agent['agentId']}")
    
    # 2. List aliases for THIS agent
    aliases = bedrock_agent_client.list_agent_aliases(agentId=agent['agentId'])
    for alias in aliases.get('agentAliasSummaries', []):
        print(f"  -> Alias Name: {alias['agentAliasName']} | Alias ID: {alias['agentAliasId']}")





Agent Name: hr-assistant-function-def | Agent ID: P58WRUHKR4
  -> Alias Name: test-alias-2 | Alias ID: 52K3N5ZT1Z
  -> Alias Name: AgentTestAlias | Alias ID: TSTALIASID
  -> Alias Name: test-alias-1 | Alias ID: ZVXQ7HQM0H


In [81]:
agentResponse = bedrock_agent_runtime_client.invoke_agent(
    inputText="Great. please reserve one day of time off for the employee with employee_id set to 1 for 2026-03-25",
    agentId=agent_id,
    agentAliasId=agent_alias_id, 
    sessionId=session_id,
    enableTrace=enable_trace, 
    endSession= end_session
)

logger.info(pprint.pprint(agentResponse))

event_stream = agentResponse['completion']
try:
    for event in event_stream:        
        if 'chunk' in event:
            data = event['chunk']['bytes']
            logger.info(f"Final answer ->\n{data.decode('utf8')}")
            agent_answer = data.decode('utf8')
            end_event_received = True
            # End event indicates that the request finished successfully
        elif 'trace' in event:
            logger.info(json.dumps(event['trace'], indent=2))
        else:
            raise Exception("unexpected event.", event)
except Exception as e:
    raise Exception("unexpected event.", e)

[2026-02-07 19:06:15,675] p74989 {3296395197.py:10} INFO - None


{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-type': 'application/vnd.amazon.eventstream',
                                      'date': 'Sun, 08 Feb 2026 03:06:15 GMT',
                                      'transfer-encoding': 'chunked',
                                      'x-amz-bedrock-agent-session-id': 'fc486b0e-049a-11f1-aa37-02211ae2cbcb',
                                      'x-amzn-bedrock-agent-content-type': 'application/json',
                                      'x-amzn-requestid': '18365088-9fbe-47dd-84e8-07a7de069b04'},
                      'HTTPStatusCode': 200,
                      'RequestId': '18365088-9fbe-47dd-84e8-07a7de069b04',
                      'RetryAttempts': 0},
 'completion': <botocore.eventstream.EventStream object at 0x1184c8850>,
 'contentType': 'application/json',
 'sessionId': 'fc486b0e-049a-11f1-aa37-02211ae2cbcb'}


[2026-02-07 19:06:17,249] p74989 {3296395197.py:17} INFO - Final answer ->
To reserve the vacation time, I need the start date for the vacation. Could you please provide the start date for the employee with employee_id 1?


In [82]:
action_group_id = agent_action_group_response['agentActionGroup']['actionGroupId']
action_group_name = agent_action_group_response['agentActionGroup']['actionGroupName']

response = bedrock_agent_client.update_agent_action_group(
    agentId=agent_id,
    agentVersion='DRAFT',
    actionGroupId= action_group_id,
    actionGroupName=action_group_name,
    actionGroupExecutor={
        'lambda': lambda_function['FunctionArn']
    },
    functionSchema={
        'functions': agent_functions
    },
    actionGroupState='DISABLED',
)

action_group_deletion = bedrock_agent_client.delete_agent_action_group(
    agentId=agent_id,
    agentVersion='DRAFT',
    actionGroupId= action_group_id
)

In [85]:
response = bedrock_agent_client.delete_agent_alias(
    agentAliasId=agent_alias_id,
    agentId=agent_id
)

ResourceNotFoundException: An error occurred (ResourceNotFoundException) when calling the DeleteAgentAlias operation: Failed to retrieve resource because it doesn't exist. Retry the request with a different resource identifier.

In [86]:
agent_deletion = bedrock_agent_client.delete_agent(
    agentId=agent_id
)

ConflictException: An error occurred (ConflictException) when calling the DeleteAgent operation: Could not delete Agent with ID P58WRUHKR4, since it has active aliases

In [87]:
lambda_client.delete_function(
    FunctionName=lambda_function_name
)

{'ResponseMetadata': {'RequestId': 'c9c56b49-8259-4ec8-8eb1-a956e89989f7',
  'HTTPStatusCode': 204,
  'HTTPHeaders': {'date': 'Sun, 08 Feb 2026 03:07:53 GMT',
   'content-type': 'application/json',
   'connection': 'keep-alive',
   'x-amzn-requestid': 'c9c56b49-8259-4ec8-8eb1-a956e89989f7'},
  'RetryAttempts': 0},
 'StatusCode': 204}

In [88]:
for policy in [agent_bedrock_allow_policy_name]:
    iam_client.detach_role_policy(RoleName=agent_role_name, PolicyArn=f'arn:aws:iam::{account_id}:policy/{policy}')
    
iam_client.detach_role_policy(RoleName=lambda_function_role, PolicyArn='arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole')

for role_name in [agent_role_name, lambda_function_role]:
    iam_client.delete_role(
        RoleName=role_name
    )

for policy in [agent_bedrock_policy]:
    iam_client.delete_policy(
        PolicyArn=policy['Policy']['Arn']
)